# Day 2 (sessions 2A + 2B): Reliable ML pipelines, cross-validation, and tuning

**Estimated core time:** 60 minutes  
**Dataset:** `data/customer_churn.csv`  
**Interview focus:** repeat the complete supervised-ML process and learn the scikit-learn syntax for validation and tuning

All executable cells begin empty. This notebook is intentionally repetitive: speed and confidence should come from rebuilding the process, not merely recognizing completed code.

## Why this matters

A model that scores well once can still be fragile. Reliable machine learning requires preprocessing that is learned inside each training fold, repeated validation, explicit hyperparameter choices, and a test set that remains untouched until the end.

For interviews, you should be able to explain both the code structure and the reason for each boundary.

## Learning objectives

By the end, you should be able to:

1. Rebuild a mixed-type preprocessing and logistic-regression pipeline.
2. Explain how `Pipeline` and `ColumnTransformer` prevent leakage.
3. Evaluate a full pipeline with stratified cross-validation.
4. Explain logistic-regression regularization and the parameter `C`.
5. Tune nested pipeline parameters with `GridSearchCV` and evaluate once on test data.

## The process to internalize

```text
raw data
  → define X and y
  → reserve untouched test set
  → define preprocessing + estimator pipeline
  → cross-validate the entire pipeline on training data
  → tune using training folds only
  → refit best configuration on all training data
  → evaluate once on test data
  → interpret errors and limitations
```

The pipeline is an estimator-like object: it can be passed directly to cross-validation and grid-search functions.

## R-to-Python bridge

| tidymodels idea | scikit-learn idea |
|---|---|
| recipe steps | transformers inside `Pipeline` / `ColumnTransformer` |
| workflow | preprocessing-plus-model `Pipeline` |
| v-fold CV | `StratifiedKFold` |
| fit resamples | `cross_validate` |
| tuning grid | `GridSearchCV` |
| finalize workflow | `best_estimator_` after refitting |
| holdout/test evaluation | predictions from `best_estimator_` on `X_test` |

The APIs differ, but the statistical separation between training, validation, and test data is the same.

## Suggested pacing

- 0–12 min: reconstruct data split and pipeline
- 12–22 min: inspect preprocessing and unknown categories
- 22–35 min: cross-validation and leakage diagnosis
- 35–48 min: regularization and grid search
- 48–60 min: final test evaluation, syntax audit, and interview explanations

If syntax recall is slow, prioritize Tasks 1, 3, 4, 6, and 7.

## Task 1 — Rebuild the ML workflow from raw data

Import the required objects, load the churn data, and create:

- `X` without `customer_id` or `churned`
- `y` containing `churned`
- numeric and categorical feature-name lists
- stratified 80/20 `X_train`, `X_test`, `y_train`, `y_test` using `random_state=42`
- numeric pipeline: median imputation, then standardization
- categorical pipeline: most-frequent imputation, then one-hot encoding with unknown categories ignored
- `preprocessor`
- `logistic_pipeline` ending in `LogisticRegression(max_iter=1000)`

New imports used later include:

```python
from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
)
```

Also import `DummyClassifier` and the accuracy, precision, recall, F1, and ROC AUC metric functions.

**Self-checks:**

- `X.shape == (2500, 10)`
- seven numeric and three categorical columns
- 2,000 training rows and 500 test rows
- do not fit anything on `X_test`

In [86]:
# Import the necessary libraries
import pandas as pd


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier

from sklearn.base import clone


# Import the data
churn = pd.read_csv('data/customer_churn.csv')

churn.head()

,customer_id,age,household_size,tenure_months,contract,internet_service,payment_method,monthly_charge_eur,monthly_usage_hours,support_calls_6m,late_payments_12m,churned
0,CH00001,49.0,2,61,monthly,none,card,69.58,56.8,2,1,1
1,CH00002,52.0,3,8,one_year,dsl,electronic_check,72.86,44.4,2,0,0
2,CH00003,37.0,1,60,monthly,fiber,card,58.54,49.3,0,0,0
3,CH00004,41.0,3,71,two_year,dsl,card,44.75,37.5,2,2,0
4,CH00005,37.0,2,6,monthly,dsl,bank_transfer,41.27,59.7,2,0,0


In [54]:
y = churn['churned']
X = churn.drop(columns = ['churned', 'customer_id'])
numeric_features = X.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X.select_dtypes(include = ['object']).columns    



X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42, stratify=y)

/var/folders/s4/hpk7skcj21v_g37ly6n069fh0000gn/T/ipykernel_35105/3938889746.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include = ['object']).columns


In [55]:
# Create the preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
    ])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])    

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
    ])


## Task 2 — Inspect what preprocessing learns

Create an independent copy and fit it for inspection:

```python
inspection_pipeline = clone(logistic_pipeline)
inspection_pipeline.fit(X_train, y_train)
```

Then inspect:

1. Outer step names.
2. Fitted numeric and categorical transformers.
3. Generated feature names with `get_feature_names_out()`.
4. Transformed training shape.
5. Numeric imputation statistics and scaling means.
6. Categories learned by the one-hot encoder.

Finally, copy one training row, replace its `contract` value with the unseen category `"three_year"`, and confirm `predict_proba` still works.

**Predict before running:** what will the encoder do with `"three_year"` when `handle_unknown="ignore"` is set?

Useful navigation pattern:

```python
pipeline.named_steps["preprocessor"].named_transformers_["numeric"]
```

In [56]:
inspection_pipeline = clone(logistic_pipeline)
inspection_pipeline.fit(X_train,y_train)
inspection_pipeline.named_steps.keys()

inspection_pipeline.named_steps['classifier'].C


1.0

In [57]:
# Inspect fitted numeric and categroical transformers
# Access preprocessor
fitted_preprocessor = inspection_pipeline.named_steps["preprocessor"]

# Access the names of each preprocessing step.
fitted_preprocessor.named_transformers_

# Acces the numeric branch
fitted_numeric = fitted_preprocessor.named_transformers_["num"]
# Acces the categorical branch
fitted_categorical = fitted_preprocessor.named_transformers_["cat"]

fitted_numeric.named_steps

{'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler()}

In [58]:
fitted_preprocessor.get_feature_names_out()

array(['num__age', 'num__household_size', 'num__tenure_months',
       'num__monthly_charge_eur', 'num__monthly_usage_hours',
       'num__support_calls_6m', 'num__late_payments_12m',
       'cat__contract_monthly', 'cat__contract_one_year',
       'cat__contract_two_year', 'cat__internet_service_dsl',
       'cat__internet_service_fiber', 'cat__internet_service_none',
       'cat__payment_method_bank_transfer', 'cat__payment_method_card',
       'cat__payment_method_electronic_check'], dtype=object)

In [60]:
# get the transform training 
fitted_preprocessor.transform(X_train).shape

(2000, 16)

In [73]:
# Get numeric imputation statistics scaling means 
fitted_numeric.named_steps['scaler'].mean_
fitted_numeric.named_steps['imputer'].statistics_

# Categories learned by the onhot encoder
fitted_categorical.named_steps['onehot'].categories_


[array(['monthly', 'one_year', 'two_year'], dtype=object),
 array(['dsl', 'fiber', 'none'], dtype=object),
 array(['bank_transfer', 'card', 'electronic_check'], dtype=object)]

## Task 3 — Cross-validate the complete pipeline

Create a reproducible splitter:

```python
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)
```

Run `cross_validate` on `logistic_pipeline` using only `X_train` and `y_train`. Request:

- ROC AUC
- precision
- recall
- training scores as well as validation scores

Scoring dictionaries use names such as:

```python
scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
}
```

Convert the returned dictionary to a `DataFrame`. Report mean and standard deviation across the five validation folds.

Then cross-validate `DummyClassifier(strategy="most_frequent")` using ROC AUC and compare it with logistic regression.

**Interpretation requirement:** explain why a distribution across folds is more informative than one score from one split. Do not touch `X_test` or `y_test` in this task.

In [82]:
# Step 1: Define the Folds
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Step 2: Define the metrics
scoring = {
    "roc_auc" : "roc_auc",
    "precision" : "precision",
    "recall" : "recall",
    "f1" : "f1"
}

# Step 3: Apply the CV
cv_results = cross_validate(
    estimator = logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# Step 4: Inspect results 
cv_results_df = pd.DataFrame(cv_results)
cv_results_df

,fit_time,score_time,test_roc_auc,train_roc_auc,test_precision,train_precision,test_recall,train_recall,test_f1,train_f1
0,0.012298,0.013900,0.786195,0.805956,0.641509,0.668317,0.357895,0.355263,0.459459,0.463918
1,0.009932,0.009885,0.806764,0.801780,0.687500,0.681592,0.347368,0.360526,0.461538,0.471601
2,0.007840,0.008417,0.810388,0.799825,0.792453,0.649746,0.442105,0.336842,0.567568,0.443674
3,0.008429,0.008825,0.794927,0.803059,0.545455,0.701923,0.315789,0.384211,0.400000,0.496599
4,0.007605,0.008110,0.768594,0.809717,0.627451,0.672811,0.336842,0.384211,0.438356,0.489112


In [85]:
cv_results_df = pd.DataFrame(cv_results).agg(["mean", "std"]).T
cv_results_df

,mean,std
fit_time,0.009221,0.001944
score_time,0.009827,0.002373
test_roc_auc,0.793374,0.016858
train_roc_auc,0.804067,0.003864
test_precision,0.658874,0.090575
train_precision,0.674878,0.019075
test_recall,0.360000,0.048467
train_recall,0.364211,0.020265
test_f1,0.465384,0.062243
train_f1,0.472981,0.020982


In [88]:
cv_dummy_results = cv_results = cross_validate(
    estimator = DummyClassifier(strategy="most_frequent"),
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

cv_dummy_results_df = pd.DataFrame(cv_dummy_results).agg(["mean", "std"]).T
cv_dummy_results_df

/Users/ericweniger/workspace/projects/learning_python/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ericweniger/workspace/projects/learning_python/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ericweniger/workspace/projects/learning_python/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this b

,mean,std
fit_time,0.000875,0.000507
score_time,0.005029,0.000964
test_roc_auc,0.500000,0.000000
train_roc_auc,0.500000,0.000000
test_precision,0.000000,0.000000
train_precision,0.000000,0.000000
test_recall,0.000000,0.000000
train_recall,0.000000,0.000000
test_f1,0.000000,0.000000
train_f1,0.000000,0.000000


## Task 4 — Diagnose leakage inside cross-validation

An AI assistant writes:

```python
transformed_training = preprocessor.fit_transform(X_train)

scores = cross_validate(
    LogisticRegression(max_iter=1000),
    transformed_training,
    y_train,
    cv=cv,
    scoring="roc_auc",
)
```

This looks safe because `X_test` is untouched, but it still leaks within cross-validation.

Answer:

1. Which rows influenced the imputation medians, scaling values, and encoded category set?
2. During a particular fold, which of those rows should have been hidden?
3. Why can the validation scores become optimistic?
4. Rewrite the call so cross-validation receives the complete `logistic_pipeline` and raw `X_train`.

**Key distinction:** the final test set can remain untouched while leakage still occurs between training and validation folds.

1. The Ai aissistant suggested to train on transormed training already. it is importent to put in the raw X and run the entire preprocessing pipeline in the cross validation. 
the preprocessed tranformed training is already one step ahaed and contains learned metrics. 
2. 
3. Because we take away the recomputing of imputation and sclaing metrics.
4. 
scores = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
)

## Task 5 — Explore logistic-regression regularization

Scikit-learn logistic regression uses regularization by default. Its `C` parameter is the inverse of regularization strength:

```text
small C → stronger regularization → coefficients pulled more strongly toward zero
large C → weaker regularization → model fits training data more freely
```

For `C` values `[0.01, 0.1, 1, 10, 100]`:

1. Clone `logistic_pipeline`.
2. set its nested parameter with `set_params(model__C=value)`.
3. Cross-validate it using the same folds and ROC AUC.
4. Store mean training AUC, mean validation AUC, and validation standard deviation in a table.

Nested parameter syntax is:

```text
pipeline_step_name__parameter_name
model__C
```

Explain what signs of underfitting or overfitting you would look for in the training-versus-validation scores. Avoid claiming meaningful differences when they are smaller than fold-to-fold variation.

In [ ]:
cloned_pipeline = clone(logistic_pipeline)
cloned_pipeline.set_params(model__C=)

## Task 6 — Tune with GridSearchCV

Create a deliberately small parameter grid:

```python
parameter_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"],
}
```

Construct `GridSearchCV` with:

- estimator: the complete `logistic_pipeline`
- `param_grid=parameter_grid`
- `scoring="roc_auc"`
- `cv=cv`
- `refit=True`
- `return_train_score=True`

Fit it using only `X_train` and `y_train`.

Inspect:

- `best_params_`
- `best_score_`
- `best_estimator_`
- selected columns from `pd.DataFrame(grid_search.cv_results_)`

**Concept questions:**

- How many parameter combinations are tested?
- How many model fits are required with five folds?
- What does `refit=True` do after choosing the best combination?
- Why is `model__C` written with two underscores?

## Task 7 — Evaluate the selected workflow once on test data

Use `grid_search.best_estimator_` to generate test-set:

- class predictions
- positive-class probabilities

Calculate accuracy, precision, recall, F1, and ROC AUC. Put them in a one-row `DataFrame` with a descriptive model name.

Compare:

- the best mean cross-validation ROC AUC
- final test ROC AUC
- precision and recall at the default threshold

**Interpretation requirement:** write three sentences:

1. How well does the model discriminate?
2. What kind of errors remain important?
3. Why must you avoid returning to grid search repeatedly after seeing the test result?

This is the first task in this notebook allowed to use `y_test` for evaluation.

## Task 8 — Syntax and object-navigation audit

Using the fitted best estimator, retrieve:

1. The fitted preprocessor.
2. The fitted categorical pipeline.
3. The fitted one-hot encoder.
4. The learned categorical levels.
5. The fitted logistic-regression model.
6. The coefficient-matrix shape.
7. The chosen value of `C`.

Use `named_steps` for pipelines and `named_transformers_` for the fitted `ColumnTransformer`.

Then explain:

- Why does the preprocessor have `transform`?
- Why does logistic regression have `predict` but not `transform`?
- Why can `GridSearchCV` accept the entire pipeline as if it were one model?

## Retrieval practice — Task 9: threshold check with little guidance

Using the best estimator's positive-class probabilities, compare thresholds 0.30 and 0.50.

For each threshold, report:

- number predicted positive
- precision
- recall

State which threshold you would prefer if missing a churner were much more expensive than contacting a customer who would have stayed.

Do not refit the model. This task changes only the probability-to-decision rule.

## Interview drill

Answer aloud or add a markdown cell:

1. Walk me through your modeling process from raw data to final evaluation.
2. Why must preprocessing occur inside each cross-validation fold?
3. What does five-fold cross-validation actually do?
4. What is regularization, and why is smaller `C` stronger in scikit-learn?
5. What does `model__C` mean?
6. What is the difference between a validation score and a test score?

A compact interview answer should mention leakage prevention, repeated estimates of generalization, tuning without touching the test set, and one final evaluation.

## Two-minute process script

Practice saying this without reading:

> I first separate predictors and target and reserve a stratified test set. I put all learned preprocessing and the estimator in one pipeline so each cross-validation fold fits imputation, scaling, encoding, and the model using only that fold's training portion. I compare fold scores and tune a small justified parameter grid on training data. Grid search refits the best configuration on all training data. I then evaluate once on the untouched test set using probabilities and metrics aligned with the decision problem.

Do not memorize the exact wording; understand the boundaries.

## Self-check and reflection

Record:

- Confidence from 1–5:
- Could you reconstruct the pipeline faster this time?
- Can you explain cross-validation without mentioning code?
- Which nested-object syntax remains confusing?
- Can you explain why preprocessing before cross-validation leaks?
- What should be repeated next?
- Approximate time spent:

Tell Codex when finished; generation alone is not completion.

## Optional extension — Visualize regularization results

Use plotnine to show `C` on a logarithmic x-axis and mean training/validation ROC AUC on the y-axis.

Explain whether the plot shows:

- underfitting at strong regularization,
- overfitting at weak regularization,
- or little practically meaningful difference.

This extension is outside the core 60 minutes.

## Finish line

You should now be able to write and explain:

**split → pipeline → stratified folds → cross-validation → nested parameters → grid search → refit → one test evaluation**

The next high-priority lesson introduces decision trees, random forests, gradient boosting, and XGBoost.